In [1]:
from dotenv import load_dotenv
from langgraph.types import interrupt, Command
import takehomeV2
load_dotenv()



API key loaded: True
Langsmith key loaded: True


True

In [4]:
examples = [
    {
        "inputs": {"severity": 1},
        "outputs": {"expected_route": "assess_case"},
    },
    {
        "inputs": {"severity": 2},
        "outputs": {"expected_route": "acknowledge_case"},
    },
    {
        "inputs": {"severity": 3},
        "outputs": {"expected_route": "acknowledge_case"},
    },
    {
        "inputs": {"severity": 4},
        "outputs": {"expected_route": "acknowledge_case"},
    },
    {
        "inputs": {"severity": 5},
        "outputs": {"expected_route": "validateSeverity"},
    },
    {
        "inputs": {"severity": 0},
        "outputs": {"expected_route": "validateSeverity"},
    },
    {
        "inputs": {"severity": "1"},
        "outputs": {"expected_route": "validateSeverity"},
    },
    {
        "inputs": {"severity": None},
        "outputs": {"expected_route": "validateSeverity"},
    },
]

In [7]:
from langsmith import Client 
client = Client()
dataset = client.create_dataset(
    dataset_name="Severity Validation new",
    description="Check valid and invalid severity routing.",
)

client.create_examples(
    dataset_id=dataset.id,
    examples=examples,
)

{'example_ids': ['ae2b3e58-2aee-4a05-8e6c-02b72165bb4e',
  '6bf0198f-917a-4c70-90e5-90d7c046132a',
  'c1398b7e-3f47-41b1-b43c-b59cf392fb22',
  'c03b7bfa-cfa4-4e36-91a3-a339d8aeedea',
  '2cfcf233-8408-4768-b1f1-14a72fc71052',
  '00b8bf50-1e9a-4519-ab50-a663d987f3b0',
  '7feec52a-4201-4381-92ac-dcfe2f19916f',
  '4b21d7c8-4b07-450a-ae92-c9ed15f518b4'],
 'count': 8,
 'as_of': '2026-11-05T20:31:11.880803627Z'}

In [25]:
def severity_target(inputs):
    route = takehomeV2.route_case(inputs)

    return {"actual_route": route}

In [26]:
def severity_evaluator(outputs, reference_outputs):
    actual = outputs["actual_route"]
    expected = reference_outputs["expected_route"]

    return {
        "key": "severity_validation",
        "score": int(actual == expected),
        "comment": f"Expected: {expected}. Actual: {actual}.",
    }

In [27]:
results = client.evaluate(
    severity_target,
    data="Severity Validation",
    evaluators=[severity_evaluator],
    experiment_prefix="severity-validation",
    max_concurrency=1,
)

View the evaluation results for experiment: 'severity-validation-ad3eb871' at:
https://smith.langchain.com/o/edf925e8-923e-4a29-86cd-32644621291e/datasets/970340b4-e938-4699-8f7a-8f69039b04dc/compare?selectedSessions=e6f2f990-1788-40a9-9e9d-c2120856aef5




In [28]:
for example in examples:
    actual = severity_target(example["inputs"])
    feedback = severity_evaluator(actual, example["outputs"])
    print(feedback)

    assert feedback["score"] == 1, feedback["comment"]

print("All severity validation checks passed.")

{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: assess_case. Actual: assess_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: acknowledge_case. Actual: acknowledge_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: acknowledge_case. Actual: acknowledge_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: acknowledge_case. Actual: acknowledge_case.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
{'key': 'severity_validation', 'score': 1, 'comment': 'Expected: validateSeverity. Actual: validateSeverity.'}
All severity validation checks passed.


In [3]:
import takehomeV2
from langsmith import Client

client = Client()

def assessment_target(inputs):
    result = takehomeV2.assess_case(inputs)
    return {"assessment": result["assessment"]}

results = client.evaluate(
    assessment_target,
    data="Severity Assessment Database",
    experiment_prefix="assessment-quality",
    max_concurrency=1,
)

View the evaluation results for experiment: 'assessment-quality-30ae1102' at:
https://smith.langchain.com/o/edf925e8-923e-4a29-86cd-32644621291e/datasets/99643d72-f2d8-41da-8e70-791e858c0df4/compare?selectedSessions=b330de65-afa8-408b-9b6d-9b87f28fb7bb




In [12]:
dataset = client.read_dataset(dataset_name="Severity Validation")

In [13]:
dataset

Dataset(name='Severity Validation', description='Check valid and invalid severity routing.', data_type=<DataType.kv: 'kv'>, id=UUID('970340b4-e938-4699-8f7a-8f69039b04dc'), created_at=datetime.datetime(2026, 11, 4, 4, 13, 0, 982253, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 11, 4, 4, 13, 0, 982253, tzinfo=TzInfo(0)), example_count=8, session_count=1, last_session_start_time=datetime.datetime(2026, 11, 4, 4, 13, 6, 981327), inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-15.3.2-arm64-arm-64bit-Mach-O', 'sdk_version': '0.14.16', 'runtime_version': '3.13.2', 'langchain_version': '1.4.7', 'py_implementation': 'CPython', 'langchain_core_version': '1.7.3'}})

In [14]:
print(dataset.url) 

https://smith.langchain.com/o/edf925e8-923e-4a29-86cd-32644621291e/datasets/970340b4-e938-4699-8f7a-8f69039b04dc
